In [2]:
import pyarrow.parquet as pq
import os 
import duckdb
import time

# User of Interests
Users who have blocked someone in the first week of September 2024 (Sept 1-7, 2024).

In [3]:
# Load the blocks database using DuckDB
blocks_path = "../../data/ale_simplicistic_model/cleaned/blocks.parquet"

# Connect to DuckDB
conn = duckdb.connect()

print("Analyzing blocks in first week of September 2024 using DuckDB...")

# Get basic info about the data
total_blocks = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{blocks_path}')").fetchone()[0]

print(f"Total blocks: {total_blocks:,}")

# Query to find blocks in the first week of September 2024 (Sept 1-7)
first_week_sept_query = f"""
WITH sept_blocks AS (
    SELECT 
        did_id,
        created_date
    FROM read_parquet('{blocks_path}')
    WHERE created_date >= '2024-09-01' 
      AND created_date < '2024-09-08'
)
SELECT
    did_id,
    COUNT(*) AS block_count
FROM sept_blocks
GROUP BY did_id
"""

# Execute query and get results
first_week_sept_blockers = conn.execute(first_week_sept_query).df()

print(f"\nUnique users who blocked someone in first week of Sept 2024: {len(first_week_sept_blockers):,}")
print(f"Total blocks in first week of Sept 2024: {first_week_sept_blockers['block_count'].sum():,}")

Analyzing blocks in first week of September 2024 using DuckDB...
Total blocks: 75,122,543

Unique users who blocked someone in first week of Sept 2024: 140,469
Total blocks in first week of Sept 2024: 1,098,207

Unique users who blocked someone in first week of Sept 2024: 140,469
Total blocks in first week of Sept 2024: 1,098,207


In [4]:
# Save the set of users who blocked someone in the first week of September 2024
sept_blockers_path = "../../data/ale_simplicistic_model/absolute/filtered/first_week_sept_blockers.parquet"

num_samples = 100_000
sept_blockers = (
    first_week_sept_blockers[['did_id']]
    .drop_duplicates()
    .reset_index(drop=True)
)
sample_size = min(len(sept_blockers), num_samples)
sept_blockers = sept_blockers.sample(n=sample_size, random_state=42).reset_index(drop=True)

sept_blockers.to_parquet(sept_blockers_path, index=False)

print(f"Saved {len(sept_blockers):,} first-week September blocker user ids to: {sept_blockers_path}")

Saved 100,000 first-week September blocker user ids to: ../../data/ale_simplicistic_model/absolute/filtered/first_week_sept_blockers.parquet


# Event-Activity Filtering
The criteria for filtering are:
- Events must involve at least one user of interest.
- Events must occur in the first week of September 2024 (Sept 1-7).

In [5]:
def filter_events(input_path, output_path, db_name, query):

    print(f"Filtering {db_name} using DuckDB...")
    start_time = time.time()
    
    conn = duckdb.connect()
    
    # Register users of interest
    users_of_interests = conn.from_df(sept_blockers)
    events_table = conn.read_parquet(input_path)
    
    # Get total row count before filtering
    total_rows = conn.execute("SELECT COUNT(*) FROM events_table").fetchall()[0][0]
    
    # Execute the provided query
    filtered_events = conn.execute(query).fetch_arrow_table()
    filtered_rows = filtered_events.num_rows
    
    # Write to parquet
    pq.write_table(filtered_events, output_path, compression='zstd')
    
    elapsed_time = time.time() - start_time
    input_size = os.path.getsize(input_path)
    output_size = os.path.getsize(output_path)
    retention_rate = (filtered_rows / total_rows * 100) if total_rows > 0 else 0
    
    print(f"\n✅ DuckDB filtering completed in {elapsed_time:.2f} seconds")
    print(f"Rows: {total_rows:,} → {filtered_rows:,} ({retention_rate:.1f}% kept)")
    print(f"File size: {input_size / (1024**3):.2f} GB → {output_size / (1024**3):.2f} GB")
    print(f"Output: {output_path}")
    
    conn.close()
    
    return {
        'elapsed_time': elapsed_time,
        'total_rows': total_rows,
        'filtered_rows': filtered_rows,
        'retention_rate': retention_rate,
        'input_size_gb': input_size / (1024**3),
        'output_size_gb': output_size / (1024**3)
    }

In [6]:
# Filter blocks in first week of September 2024 (two-user events) 
blocks_input_path = "../../data/ale_simplicistic_model/cleaned/blocks.parquet"
blocks_output_path = "../../data/ale_simplicistic_model/absolute/filtered/blocks.parquet"

blocks_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi_act ON e.did_id = uoi_act.did_id
LEFT JOIN users_of_interests uoi_sub ON e.subject_did_id = uoi_sub.did_id
WHERE 
    e.created_date >= '2024-09-01' 
    AND e.created_date < '2024-09-08'
    AND (
        uoi_act.did_id IS NOT NULL 
        OR uoi_sub.did_id IS NOT NULL
    )
"""

stats_blocks = filter_events(blocks_input_path, blocks_output_path, "blocks", blocks_query)
print(f"Blocks filtered: {stats_blocks['filtered_rows']:,} blocks from {stats_blocks['total_rows']:,} total")

Filtering blocks using DuckDB...

✅ DuckDB filtering completed in 1.52 seconds
Rows: 75,122,543 → 808,368 (1.1% kept)
File size: 0.31 GB → 0.00 GB
Output: ../../data/ale_simplicistic_model/absolute/filtered/blocks.parquet
Blocks filtered: 808,368 blocks from 75,122,543 total


In [ ]:
# Filter posts in first week of September 2024 (single-user events)
posts_input_path = "../../data/ale_simplicistic_model/cleaned/posts.parquet"
posts_output_path = "../../data/ale_simplicistic_model/absolute/filtered/posts.parquet"

posts_query = """
SELECT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi ON e.did_id = uoi.did_id
WHERE e.created_date >= '2024-09-01' 
  AND e.created_date < '2024-09-08'
  AND uoi.did_id IS NOT NULL
"""

stats_posts = filter_events(posts_input_path, posts_output_path, "posts", posts_query)
print(f"Posts filtered: {stats_posts['filtered_rows']:,} posts from {stats_posts['total_rows']:,} total")

In [ ]:
# Filter follows in first week of September 2024 (two-user events)
follows_input_path = "../../data/ale_simplicistic_model/cleaned/follows.parquet"
follows_output_path = "../../data/ale_simplicistic_model/absolute/filtered/follows.parquet"

follows_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi_act ON e.did_id = uoi_act.did_id
LEFT JOIN users_of_interests uoi_sub ON e.subject_did_id = uoi_sub.did_id
WHERE 
    e.created_date >= '2024-09-01' 
    AND e.created_date < '2024-09-08'
    AND (
        uoi_act.did_id IS NOT NULL 
        OR uoi_sub.did_id IS NOT NULL
    )
"""

stats_follows = filter_events(follows_input_path, follows_output_path, "follows", follows_query)
print(f"Follows filtered: {stats_follows['filtered_rows']:,} follows from {stats_follows['total_rows']:,} total")

In [6]:
# Filter likes in first week of September 2024 (two-user events)
likes_input_path = "../../data/ale_simplicistic_model/cleaned/likes.parquet"
likes_output_path = "../../data/ale_simplicistic_model/absolute/filtered/likes.parquet"

likes_query = """
SELECT DISTINCT e.*
FROM events_table e
LEFT JOIN users_of_interests uoi_act ON e.did_id = uoi_act.did_id
LEFT JOIN users_of_interests uoi_sub ON e.subject_did_id = uoi_sub.did_id
WHERE 
    e.created_date >= '2024-09-01' 
    AND e.created_date < '2024-09-08'
    AND (
        uoi_act.did_id IS NOT NULL 
        OR uoi_sub.did_id IS NOT NULL
    )
"""

stats_likes = filter_events(likes_input_path, likes_output_path, "likes", likes_query)
print(f"Likes filtered: {stats_likes['filtered_rows']:,} likes from {stats_likes['total_rows']:,} total")

Filtering likes using DuckDB...

✅ DuckDB filtering completed in 43.01 seconds
Rows: 3,867,932,497 → 31,487,780 (0.8% kept)
File size: 9.12 GB → 0.15 GB
Output: ../../data/ale_simplicistic_model/absolute/filtered/likes.parquet

✅ DuckDB filtering completed in 43.01 seconds
Rows: 3,867,932,497 → 31,487,780 (0.8% kept)
File size: 9.12 GB → 0.15 GB
Output: ../../data/ale_simplicistic_model/absolute/filtered/likes.parquet
Likes filtered: 31,487,780 likes from 3,867,932,497 total
Likes filtered: 31,487,780 likes from 3,867,932,497 total
